# Results viewer

Training runs from `runs/`, read-only. **Run All** (⌘⇧↩ in the toolbar menu) to refresh; kernel: `.venv` (Python 3.14).

- **Gold** macro AUC (58 studies, pooled out-of-fold) is the arbiter. Differences below about **0.02 are noise**.
- **Report** macro AUC (4,349 report-only studies) is the steadier signal for choosing between models.
- Smoke runs (fewer than 5 folds) are hidden; set `INCLUDE_SMOKE = True` to see them.
- Metrics only. This notebook never loads report text or images.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/knee").exists())
sys.path.insert(0, str(ROOT / "src"))
import importlib
from IPython.display import display
from knee import results as R
importlib.reload(R)  # pick up edits to results.py without restarting the kernel
%matplotlib inline

INCLUDE_SMOKE = False
BASELINE = None   # None = first finished run described "baseline…"; or a run id, e.g. "20260929-134754"
CANDIDATE = None  # None = latest finished run

runs = R.load_runs(include_smoke=INCLUDE_SMOKE)
BASE = R.pick_baseline(runs, BASELINE)
CAND = R.pick_candidate(runs, CANDIDATE, BASE)
print(f"{len(runs)} runs · baseline {BASE} · candidate {CAND}")

## In progress
Runs without `metrics.json` yet. A run with no new epoch for 20 min shows as *stopped*.

In [ ]:
fig = R.live(runs)

## Leaderboard
Every run with its change, deltas against the baseline and the three weakest gold labels.

In [ ]:
R.leaderboard(runs, BASE)

## Per label
AUC per label for each finished run; colour is the difference from the baseline (blue better, red worse).

In [ ]:
display(R.label_table(runs, "report", BASE))
display(R.label_table(runs, "gold", BASE))

## Baseline vs candidate
Paired bootstrap over studies (the same resample for both runs): the 90% interval shows how much of the difference could be sampling noise.

In [ ]:
if CAND:
    display(R.paired_bootstrap(runs, BASE, CAND).style.format({"studies": "{:.0f}", "diff": "{:+.4f}", "lo 5%": "{:+.4f}",
                                                              "hi 95%": "{:+.4f}", "P(better)": "{:.2f}"}))
    R.plot_dumbbell(runs, BASE, CAND)
    R.plot_curves(runs, [BASE, CAND])
else:
    print("Needs two finished runs.")

## Knee side
Report AUC for right vs left knees. Findings that sit on one side of the knee (menisci, tibiofemoral OA, MCL, Baker's) land on opposite image sides for left and right knees. A model that has not learnt the side scores lower on one side for them. First finding (PCA run): MCL 0.06 lower on left knees; menisci/OA no gap. Canonicalisation (`--canon`) should close it.

In [ ]:
for r in [BASE, CAND]:
    if r:
        t = R.side_split(runs, r)
        print(r, "·", runs.loc[r, "change"])
        display(t.style.format({c: "{:+.3f}" if c == "L − R" else "{:.3f}" for c in t.columns if c != "group"}))
        print(t.groupby("group")["L − R"].mean().round(3).to_dict())